In [2]:
import numpy as np
import pandas as pd

In [3]:
np.random.seed(0)
cov_input = pd.read_csv("/Users/fuyuxuan/Downloads/test5_2.csv")

# Convert it to numpy array
cov_matrix = cov_input.values

# Symmetrize the covariance matrix (or sigma)
cov_matrix = 0.5 * (cov_matrix + cov_matrix.T)

# Eigen decomposition covariance matrix
lam, V = np.linalg.eigh(cov_matrix)

# Sort eigenvalues/vectors descending
idx = np.argsort(lam)[::-1]
lam = lam[idx]
V = V[:, idx]

# Choose smallest k such that >= 99% variance explained
total_var = lam.sum()
cum_var = np.cumsum(lam) / (total_var + 1e-30)
k = int(np.searchsorted(cum_var, 0.99) + 1)

# Keep top-k components
lam_k = lam[:k]
V_k = V[:, :k]

# PCA simulation with k factors: Generate Z ~ N(0, I_k) with 100000 x k, then X = Z * sqrt(lam_k) * V_k^T
# This will give us Cov(X) = V_k diag(lam_k) V_k^T
number_of_simulation = 100000
Z = np.random.normal(size=(number_of_simulation, k))
Z_scaled = Z * np.sqrt(lam_k)
X = Z_scaled @ V_k.T

# Compute output covariance matrix
cov_output = np.cov(X, rowvar=False, ddof=0)

# Convert it to DataFrame
cov_output_df = pd.DataFrame(cov_output, columns=cov_input.columns, index=cov_input.columns)

print(cov_output_df)

          x1        x2        x3        x4        x5
x1  0.084715  0.116419  0.042059  0.008958  0.003862
x2  0.116419  0.159986  0.057799  0.012310  0.005307
x3  0.042059  0.057799  0.037177  0.005983  0.002568
x4  0.008958  0.012310  0.005983  0.001092  0.000470
x5  0.003862  0.005307  0.002568  0.000470  0.000202
